# Stage 1 CPU preparation and validation
Run cells in order. Mount the intended Drive account. This notebook calls shared Python modules under `src/stage1/`; it does not require the removed Stage 1 `.py` entry scripts. Failure or blocked status stops the notebook. Keep the checkout clean; outputs belong under PHASE1_ROOT, never the repository. Actual Drive sync and model readiness require their own checks.
Use a CPU runtime. Run this notebook once for both models and all four languages.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO = Path('/content/Research')
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
REPO_URL = 'https://github.com/Drazr/Research.git'
BRANCH = 'phase1'
STORAGE_SCOPE = 'persistent-drive'
DATASET_MANIFEST = None
PREPARATION_COMMIT = None


## Mount Drive and clone phase1
Existing checkouts are not reset or updated automatically. Outside Colab, set explicit paths and local-development scope.


In [ ]:
try:
    from google.colab import drive
except ImportError:
    if STORAGE_SCOPE != 'local-development':
        raise RuntimeError('Outside Colab: configure an explicit local root and local-development scope.')
else:
    drive.mount('/content/drive')
    if not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('Drive mount is unavailable.')

if not (REPO / '.git').exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError('Repository destination is nonempty; choose an empty path.')
    REPO.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO)], check=True)

def git_output(*args):
    return subprocess.check_output(['git', '-C', str(REPO), *args], text=True).strip()

if git_output('remote', 'get-url', 'origin').removesuffix('.git').rstrip('/') != REPO_URL.removesuffix('.git'):
    raise RuntimeError('Expected the Drazr/Research repository.')
if git_output('branch', '--show-current') != BRANCH:
    raise RuntimeError('Select the phase1 branch before continuing.')
if git_output('status', '--porcelain'):
    raise RuntimeError('Use a clean checkout so the recorded code commit matches the scripts.')
CODE_COMMIT = git_output('rev-parse', 'HEAD')
PHASE1_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['PHASE1_ROOT'] = str(PHASE1_ROOT)
sys.path.insert(0, str(REPO))
print('Code commit:', CODE_COMMIT)
print('Storage root:', PHASE1_ROOT)


## Install CPU requirements and define runner


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements-data.txt')], check=True)

COMMON_ARGS = [
    '--phase1-root', str(PHASE1_ROOT),
    '--code-commit', CODE_COMMIT,
    '--storage-scope', STORAGE_SCOPE,
]

def run_stage1_task(name, extra_args=()):
    if git_output('rev-parse', 'HEAD') != CODE_COMMIT or git_output('status', '--porcelain'):
        raise RuntimeError('Repository changed after setup; rerun setup from a clean checkout.')
    task_args = list(COMMON_ARGS)
    if name == 'validate_model.py':
        index = task_args.index('--code-commit')
        del task_args[index:index+2]
    command = [sys.executable, '-m', 'src.stage1.' + name.removesuffix('.py'), *task_args, *extra_args]
    report = None
    with subprocess.Popen(command, cwd=str(REPO), stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
            try:
                candidate = json.loads(line)
            except json.JSONDecodeError:
                continue
            if isinstance(candidate, dict) and 'status' in candidate:
                report = candidate
        returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(f'{name} failed (exit {returncode}); fix the error before continuing.')
    if report is None or report.get('status') != 'passed':
        raise RuntimeError(f'{name} did not return a passed report.')
    return report


## Prepare dataset


In [ ]:
# Clear a previous success before starting a new attempt.
DATASET_MANIFEST = None
PREPARATION_COMMIT = None
preparation_report = run_stage1_task('prepare_dataset.py')
candidate = Path(preparation_report['dataset_manifest']).resolve()
candidate.relative_to(PHASE1_ROOT.resolve())
if candidate.name != '_COMPLETE.json' or not candidate.is_file():
    raise RuntimeError('Preparation did not produce its completion manifest.')
DATASET_MANIFEST = candidate
PREPARATION_COMMIT = CODE_COMMIT
print('Dataset manifest:', DATASET_MANIFEST)


## Validate prepared dataset


In [ ]:
if DATASET_MANIFEST is None or PREPARATION_COMMIT != CODE_COMMIT:
    raise RuntimeError('Run the preparation cell successfully for this code commit first.')
validation_report = run_stage1_task(
    'validate_data.py', ['--dataset-manifest', str(DATASET_MANIFEST)]
)
print('CPU preparation and validation passed.')
print('Dataset manifest for later model checks:', DATASET_MANIFEST)
print('Validation report:', validation_report['report'])
